                                           ***TUGAS MANDIRI***

***MENYIAPKAN DATASET DAN SPARK SESSION***

In [6]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
import numpy as np, pandas as pd

spark = SparkSession.builder.appName("Tugas12").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Membuat dataset (sesuai instruksi)
np.random.seed(212)
n = 3000
data_tugas12 = pd.DataFrame({
    "nama_menu": np.random.choice(["Kopi Susu","Americano","Cappuccino","Roti Bakar","Kentang Goreng","Es Teh"], size=n),
    "cabang": np.random.choice(["Magelang","Yogyakarta","Semarang"], size=n),
    "waktu_pesan": np.random.choice(["Pagi","Siang","Sore","Malam"], size=n),
    "qty": np.random.randint(1, 6, size=n),
    "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
})
prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)
data_tugas12.to_csv("tugas12_ulasan.csv", index=False)

df = spark.read.csv("tugas12_ulasan.csv", header=True, inferSchema=True)
df.show(5)
df.groupBy("ulasan_positif").count().show()

+--------------+----------+-----------+---+------------------+--------------+
|     nama_menu|    cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|
+--------------+----------+-----------+---+------------------+--------------+
|     Kopi Susu|Yogyakarta|       Sore|  5|                 5|             1|
|        Es Teh|Yogyakarta|       Pagi|  1|                21|             0|
|    Cappuccino|Yogyakarta|       Sore|  5|                15|             0|
|Kentang Goreng|  Semarang|      Malam|  2|                 5|             1|
|     Americano|Yogyakarta|      Malam|  2|                10|             0|
+--------------+----------+-----------+---+------------------+--------------+
only showing top 5 rows

+--------------+-----+
|ulasan_positif|count|
+--------------+-----+
|             1| 1390|
|             0| 1610|
+--------------+-----+



In [7]:
import numpy as np
import pandas as pd

np.random.seed(212)
n = 3000
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas12 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
})

# Simulasi: waktu tunggu lebih lama cenderung (namun tidak selalu) menurunkan peluang ulasan positif
prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)

data_tugas12.to_csv("tugas12_ulasan.csv", index=False)
print(f"Dataset berhasil dibuat: {len(data_tugas12)} baris")
print("Distribusi label:")
print(data_tugas12["ulasan_positif"].value_counts())

Dataset berhasil dibuat: 3000 baris
Distribusi label:
ulasan_positif
0    1610
1    1390
Name: count, dtype: int64


***A. MEMBANGUN ML PIPELINE***

In [8]:
idx_menu   = StringIndexer(inputCol="nama_menu",   outputCol="nama_menu_idx")
idx_cabang = StringIndexer(inputCol="cabang",      outputCol="cabang_idx")
idx_waktu  = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")

ohe = OneHotEncoder(inputCols=["nama_menu_idx","cabang_idx","waktu_pesan_idx"],
                    outputCols=["nama_menu_vec","cabang_vec","waktu_pesan_vec"])

assembler = VectorAssembler(
    inputCols=["nama_menu_vec","cabang_vec","waktu_pesan_vec","qty","waktu_tunggu_menit"],
    outputCol="features")

lr = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")

fe_stages = [idx_menu, idx_cabang, idx_waktu, ohe, assembler]   # dipakai ulang di bagian C
pipeline_lr = Pipeline(stages=fe_stages + [lr])
print([type(s).__name__ for s in pipeline_lr.getStages()])

['StringIndexer', 'StringIndexer', 'StringIndexer', 'OneHotEncoder', 'VectorAssembler', 'LogisticRegression']


***B. PELATIHAN DAN EVALUASI***

In [9]:
train, test = df.randomSplit([0.8, 0.2], seed=77)
model_lr = pipeline_lr.fit(train)
pred_lr = model_lr.transform(test)

ev_acc = MulticlassClassificationEvaluator(labelCol="ulasan_positif", metricName="accuracy")
ev_f1  = MulticlassClassificationEvaluator(labelCol="ulasan_positif", metricName="f1")
ev_auc = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")

acc_lr, f1_lr, auc_lr = ev_acc.evaluate(pred_lr), ev_f1.evaluate(pred_lr), ev_auc.evaluate(pred_lr)
print(f"Accuracy: {acc_lr:.4f} | F1: {f1_lr:.4f} | AUC-ROC: {auc_lr:.4f}")

print("Confusion matrix:")
pred_lr.groupBy("ulasan_positif","prediction").count().orderBy("ulasan_positif","prediction").show()

Accuracy: 0.7929 | F1: 0.7926 | AUC-ROC: 0.8650
Confusion matrix:
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



***C. MODEL PEMBANDING***

In [10]:
rf = RandomForestClassifier(featuresCol="features", labelCol="ulasan_positif", numTrees=30, seed=77)
model_rf = Pipeline(stages=fe_stages + [rf]).fit(train)     # 5 tahap identik, hanya tahap akhir diganti
pred_rf = model_rf.transform(test)

acc_rf, f1_rf, auc_rf = ev_acc.evaluate(pred_rf), ev_f1.evaluate(pred_rf), ev_auc.evaluate(pred_rf)

print(pd.DataFrame({"Model": ["Logistic Regression","Random Forest"],
                    "Accuracy": [acc_lr, acc_rf], "F1": [f1_lr, f1_rf], "AUC": [auc_lr, auc_rf]}).round(4).to_string(index=False))

              Model  Accuracy     F1    AUC
Logistic Regression    0.7929 0.7926 0.8650
      Random Forest    0.7963 0.7959 0.8455


***D. ANALISIS KESALAHAN MODEL*** 

In [11]:
nama_terbaik, pred_terbaik = ("Random Forest", pred_rf) if auc_rf > auc_lr else ("Logistic Regression", pred_lr)
print("Model terbaik:", nama_terbaik)

salah = pred_terbaik.filter(col("ulasan_positif") != col("prediction"))
print(f"Salah: {salah.count()} dari {pred_terbaik.count()}")

# 10 baris salah (sampel acak, karena hasil randomSplit terurut per partisi)
salah.orderBy(F.rand(seed=77)).select("nama_menu","cabang","waktu_pesan","qty","waktu_tunggu_menit",
                                       "ulasan_positif","prediction").show(10, truncate=False)

# Pola: error rate per waktu tunggu
(pred_terbaik.withColumn("salah", (col("ulasan_positif") != col("prediction")).cast("int"))
    .groupBy("waktu_tunggu_menit").agg(F.count("*").alias("jumlah"), F.round(F.avg("salah"),3).alias("error_rate"))
    .orderBy("waktu_tunggu_menit").show(25))

Model terbaik: Logistic Regression
Salah: 122 dari 589
+--------------+----------+-----------+---+------------------+--------------+----------+
|nama_menu     |cabang    |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|
+--------------+----------+-----------+---+------------------+--------------+----------+
|Cappuccino    |Magelang  |Malam      |1  |8                 |0             |1.0       |
|Es Teh        |Magelang  |Pagi       |3  |10                |0             |1.0       |
|Kentang Goreng|Magelang  |Sore       |1  |13                |1             |0.0       |
|Roti Bakar    |Magelang  |Sore       |3  |6                 |0             |1.0       |
|Es Teh        |Yogyakarta|Malam      |5  |12                |0             |1.0       |
|Es Teh        |Magelang  |Siang      |2  |13                |1             |0.0       |
|Kentang Goreng|Magelang  |Pagi       |1  |4                 |0             |1.0       |
|Roti Bakar    |Yogyakarta|Sore       |1  |16          

***E. KESIMPULAN BISNIS***

In [12]:
# Pendukung E: fitur paling berpengaruh
attrs = pred_lr.schema["features"].metadata["ml_attr"]["attrs"]
nama = [a["name"] for _, v in sorted([(a["idx"], a) for k in attrs for a in attrs[k]]) for a in [v]]
fi = pd.DataFrame({"fitur": nama, "koef_LR": model_lr.stages[-1].coefficients.toArray(),
                   "importance_RF": model_rf.stages[-1].featureImportances.toArray()})
print(fi.sort_values("importance_RF", ascending=False).round(4).to_string(index=False))
print("Odds ratio per +1 menit tunggu:", round(float(np.exp(fi.loc[fi.fitur=="waktu_tunggu_menit","koef_LR"].iloc[0])), 3))

                   fitur  koef_LR  importance_RF
      waktu_tunggu_menit  -0.2220         0.9338
                     qty  -0.0108         0.0126
    waktu_pesan_vec_Sore   0.1857         0.0088
   waktu_pesan_vec_Malam   0.2530         0.0084
 nama_menu_vec_Kopi Susu   0.1171         0.0083
     cabang_vec_Semarang  -0.1140         0.0060
    waktu_pesan_vec_Pagi  -0.0651         0.0045
 nama_menu_vec_Americano  -0.0855         0.0045
    nama_menu_vec_Es Teh  -0.0191         0.0041
   cabang_vec_Yogyakarta   0.0645         0.0033
nama_menu_vec_Roti Bakar  -0.1443         0.0032
nama_menu_vec_Cappuccino   0.0194         0.0025
Odds ratio per +1 menit tunggu: 0.801


                                            ***EKSPLORASI***

In [15]:
# Kelompokkan waktu tunggu ke 4 rentang agar mudah dibaca
df2 = df.withColumn("rentang_tunggu",
        F.when(F.col("waktu_tunggu_menit") <= 6, "1. 2-6 menit")
         .when(F.col("waktu_tunggu_menit") <= 12, "2. 7-12 menit")
         .when(F.col("waktu_tunggu_menit") <= 18, "3. 13-18 menit")
         .otherwise("4. 19-24 menit"))

df2.groupBy("rentang_tunggu").agg(F.count("*").alias("jumlah"),
        F.round(F.avg("ulasan_positif"), 3).alias("rasio_positif")).orderBy("rentang_tunggu").show()

print("Korelasi waktu_tunggu vs ulasan_positif:", round(df.stat.corr("waktu_tunggu_menit", "ulasan_positif"), 3))
print("Korelasi qty vs ulasan_positif         :", round(df.stat.corr("qty", "ulasan_positif"), 3))

+--------------+------+-------------+
|rentang_tunggu|jumlah|rasio_positif|
+--------------+------+-------------+
|  1. 2-6 menit|   642|        0.871|
| 2. 7-12 menit|   766|        0.642|
|3. 13-18 menit|   766|        0.313|
|4. 19-24 menit|   826|         0.12|
+--------------+------+-------------+

Korelasi waktu_tunggu vs ulasan_positif: -0.581
Korelasi qty vs ulasan_positif         : 0.0


In [14]:
# rata-rata kolom 0/1 = rasio ulasan positif
for kolom in ["nama_menu", "cabang", "waktu_pesan"]:
    df.groupBy(kolom).agg(F.round(F.avg("ulasan_positif"), 3).alias("rasio_positif")).orderBy(kolom).show()

+--------------+-------------+
|     nama_menu|rasio_positif|
+--------------+-------------+
|     Americano|        0.478|
|    Cappuccino|        0.477|
|        Es Teh|        0.436|
|Kentang Goreng|         0.47|
|     Kopi Susu|        0.487|
|    Roti Bakar|        0.429|
+--------------+-------------+

+----------+-------------+
|    cabang|rasio_positif|
+----------+-------------+
|  Magelang|        0.459|
|  Semarang|        0.452|
|Yogyakarta|        0.477|
+----------+-------------+

+-----------+-------------+
|waktu_pesan|rasio_positif|
+-----------+-------------+
|      Malam|        0.478|
|       Pagi|        0.449|
|      Siang|        0.455|
|       Sore|        0.472|
+-----------+-------------+

